# Run an experiment job on Colab

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then set the parameters in the next cell and choose *Runtime → Run all*.

Results go to Google Drive (`MyDrive/kasukabe/<JOB>/`), so a disconnect loses at most the session in progress. After a disconnect, just *Run all* again: the job resumes.

## 1. Parameters

In [ ]:
JOB = "j1_parse"        # j1_parse, j2_baseline or j3_prelim
SHARD = "1/1"           # this run's slice of the job, e.g. "1/2"
REF = "jobs-v1"         # git tag to run: jobs-v1 for J1/J2, jobs-v2 for J3
PARALLEL = 8            # sessions at once
LIMIT = None            # e.g. 2 for a quick test run; None for the full job
BACKEND = "ollama"      # "ollama" (default, reliable on a T4) or "vllm"

## 2. GPU check

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
assert gpu.returncode == 0, "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again."

## 3. Google Drive (results live here)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/kasukabe/{JOB}"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 4. Code at the chosen tag, and its Python environment

In [ ]:
%%bash -s "$REF"
set -e
REPO=/content/repo
if [ ! -d $REPO/.git ]; then
  git clone -q https://github.com/liteleliya/Reasoning-AI-Agents---Kasukabe-Defence-Group.git $REPO
fi
cd $REPO
git fetch -q --tags origin
git checkout -q "$1"
git log -1 --oneline
command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH=$HOME/.local/bin:$PATH
uv sync -q
uv run python -c "import bench.run, agents.pex_agent; print('code OK')"

## 5. Model server

Ollama serves `qwen2.5:7b-instruct` (about 3 minutes to install and download the first time). The vLLM option is kept for later, faster runs.

In [ ]:
import os, subprocess, time, urllib.request
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

def wait_for(url, minutes=10):
    for _ in range(minutes * 12):
        try:
            urllib.request.urlopen(url, timeout=5)
            return True
        except Exception:
            time.sleep(5)
    return False

if BACKEND == "ollama":
    MODEL, BASE_URL = "qwen2.5:7b-instruct", "http://127.0.0.1:11434/v1"
    if subprocess.run("command -v ollama", shell=True).returncode != 0:
        subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)
    env = {**os.environ, "OLLAMA_NUM_PARALLEL": str(PARALLEL), "OLLAMA_CONTEXT_LENGTH": "8192",
           "OLLAMA_HOST": "127.0.0.1:11434"}
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/server.log", "w"),
                     stderr=subprocess.STDOUT)
    assert wait_for("http://127.0.0.1:11434/api/version", 2), open("/content/server.log").read()[-3000:]
    subprocess.run(["ollama", "pull", MODEL], check=True)
else:
    MODEL, BASE_URL = "Qwen/Qwen2.5-7B-Instruct-AWQ", "http://127.0.0.1:8001/v1"
    subprocess.run("pip install -q vllm", shell=True, check=True)
    subprocess.Popen(["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
                      "--quantization", "awq", "--dtype", "half", "--max-model-len", "8192",
                      "--gpu-memory-utilization", "0.90", "--port", "8001"],
                     stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
    assert wait_for("http://127.0.0.1:8001/v1/models", 15), open("/content/server.log").read()[-3000:]

os.environ.update(LLM_BACKEND="openai", LLM_BASE_URL=BASE_URL, LLM_MODEL=MODEL)
print("model server ready:", MODEL)

## 6. Smoke test: one call to the model

In [ ]:
!cd /content/repo && uv run python -m agents.llm_client --json "Reply in JSON with keys answer and reason: what is 2+2?"

## 7. Run the job

Progress lines look like `done 37/120 · last session 41 s · consensus · eta 0h52m`. The summary at the end is what to report.

In [ ]:
import shlex
if JOB == "j1_parse":
    cmd = f"uv run python -m bench.parse_check --n 50 --parallel {PARALLEL} --out {shlex.quote(OUT)}"
else:
    cmd = (f"uv run python -m bench.run configs/{JOB}.json --shard {SHARD} --parallel {PARALLEL} "
           f"--out {shlex.quote(OUT)}" + (f" --limit {LIMIT}" if LIMIT else ""))
print(cmd)
!cd /content/repo && {cmd} 2>&1 | tee -a /content/job.log

## 8. If something failed

Copy everything this cell prints and send it to Aadi.

In [ ]:
!tail -n 40 /content/job.log; echo ----- server -----; tail -n 40 /content/server.log; echo ----- files -----; ls -la "$OUT" "$OUT"/shard-* 2>/dev/null